<a href="https://colab.research.google.com/github/Farrukh776/flyrank-ai/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Farrukh776/flyrank-ai/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding #2 — "The Content Performance Curve" (health score peaks at 61-90 days, hits a "decay cliff" at 271-365 days, then "recovers" at 365+)

This finding compares the current health score of different pages grouped by their current age — 0-7 days, 8-14 days, ... 365+ days — at one point in time. Methodology question: is this a longitudinal pattern (the same pages tracked as they age) or a cross-sectional snapshot (different pages that happen to be different ages today)? The paper's own language — "content enters a growth phase," "hits peak performance," "the decay cliff" — reads like a single page's lifecycle, but the underlying table is cross-sectional. If it's cross-sectional, the "decay cliff" at 271-365 days could partly reflect which pages survive to be that age (e.g., older cohorts might include a different mix of topics or original quality) rather than any single page's health actually falling over time. The paper does soften the 365+ "recovery" finding responsibly ("this is not evidence that age naturally reverses decline on its own"), which is exactly the right instinct — I'd just ask for that same caveat to extend to the whole curve, not just the recovery tail.

Finding #4 — "The Freshness Multiplier" (growth-to-decline ratio by freshness window; the paper itself flags the 361+ bucket as "too small and too unstable," 283:1 from just 1 declining page)

Methodology question: where does the underlying trend_direction label (up/down/stable) actually come from? Per the paper's own metrics glossary, it's a single 30-day-vs-previous-30-day impression comparison, with >10%/<-10% thresholds. That's a fairly noisy, single-window definition — a page could cross the ±10% line from ordinary week-to-week variance rather than a genuine structural trend, especially for lower-traffic pages. The paper already models excellent practice here by explicitly flagging the 361+ cell as unreliable (n=1) rather than hiding it — I'd extend the same scrutiny one step further and ask whether the 31-90d "strongest growth window" figure (7.88:1) would hold up under a longer or rolling-average trend definition, rather than a single 30-vs-30 snapshot.


In [ ]:
#setup

import os, subprocess, sys
if "google.colab" in sys.modules and not os.path.exists("flyrank-ai"):
    subprocess.run(["git", "clone", "https://github.com/Farrukh776/flyrank-ai.git"], check=True)
if os.path.basename(os.getcwd()) != "flyrank-ai":
    os.chdir("flyrank-ai")

%pip -q install duckdb
import duckdb, pandas as pd, numpy as np
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

features = con.sql(f"""
SELECT
  f.content_hash_id, f.client_hash_id,
  SUM(f.gsc_impressions) AS impressions_month,
  SUM(f.gsc_clicks) AS clicks_month,
  SUM(f.gsc_sum_position) / NULLIF(SUM(f.gsc_impressions), 0) AS avg_position,
  DATE '2026-03-31' - ANY_VALUE(d.content_created_date) AS content_age_days,
  ANY_VALUE(d.word_count) AS word_count,
  ANY_VALUE(d.search_volume) AS search_volume,
  ANY_VALUE(d.competition) AS competition
FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet') f
LEFT JOIN read_parquet('{REL}/dim_content.parquet') d
  ON f.content_hash_id = d.content_hash_id
GROUP BY f.content_hash_id, f.client_hash_id
""").df()
features["ctr"] = features["clicks_month"] / features["impressions_month"].replace(0, np.nan)

halves = con.sql(f"""
SELECT content_hash_id,
  SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_impressions END) AS impr_h1,
  SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions END) AS impr_h2
FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')
GROUP BY content_hash_id
""").df()
halves["pct_change"] = (halves["impr_h2"] - halves["impr_h1"]) / halves["impr_h1"].replace(0, pd.NA)
halves["declined"] = (halves["pct_change"] < 0).astype(int)

df = features.merge(halves[["content_hash_id", "declined"]], on="content_hash_id")
df = df.dropna(subset=["avg_position", "ctr"])

feature_cols = ["impressions_month", "avg_position", "ctr", "content_age_days",
                 "word_count", "search_volume", "competition"]
X = df[feature_cols].fillna(0)
y = df["declined"]
groups = df["client_hash_id"]

print(df.shape, "| decline base rate:", round(y.mean(), 3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(176738, 11) | decline base rate: 0.377


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Before (random split): 0.85 @20, 0.86 @50. After (grouped by client): 0.55 @20, 0.66 @50.

That's a ~30-point gap at precision@20 — a large, honest confession. The random split let the model partially "memorize" client identity: since content from the same client shares hidden characteristics (site design, niche, backlink profile, editorial habits), seeing some of a client's pages in training let the model shortcut its way to good scores on that same client's other pages in test, without truly generalizing. Per hunting-leakage-and-validating, this gap is the finding — it quantifies how much of the "skill" in a naive random-split evaluation was actually memorization rather than a transferable pattern. The grouped number (0.55/0.66) is the one that should be trusted and reported going forward — it reflects performance on clients the model never saw during training, which is the honest question for a real deployment.

In [ ]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# "BEFORE" — random split, ignores that pages repeat within clients
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
rf_random = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42)
rf_random.fit(X_tr_r, y_tr_r)
score_random = rf_random.predict_proba(X_te_r)[:, 1]

# "AFTER" — grouped split (same as Week 5)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_tr_g, X_te_g = X.iloc[train_idx], X.iloc[test_idx]
y_tr_g, y_te_g = y.iloc[train_idx], y.iloc[test_idx]
rf_grouped = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42)
rf_grouped.fit(X_tr_g, y_tr_g)
score_grouped = rf_grouped.predict_proba(X_te_g)[:, 1]

before_after = pd.DataFrame({
    "split": ["random (before)", "grouped by client (after)"],
    "precision_at_20": [precision_at_k(score_random, y_te_r.values, 20),
                         precision_at_k(score_grouped, y_te_g.values, 20)],
    "precision_at_50": [precision_at_k(score_random, y_te_r.values, 50),
                         precision_at_k(score_grouped, y_te_g.values, 50)],
})
before_after

,split,precision_at_20,precision_at_50
0,random (before),0.85,0.86
1,grouped by client (after),0.55,0.66


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Attack checklist: no label-derived features, no product flags, grouped split confirmed, base rate printed (0.377) next to the metric (0.66 @50 — real, substantial skill above chance).

With/without top feature test: removing content_age_days — the top feature by permutation importance in Week 5 — barely changes precision@50 (0.66 → 0.68, actually slightly higher without it). This is a reassuring, not alarming, result: per the leakage taxonomy, a collapse toward the base rate when removing a suspect feature is the signature of real leakage (the model was just reading that one column). Here, the score holding steady — or even improving slightly — without it means content_age_days was contributing real but non-dominant signal alongside the other features, not secretly encoding the label. Combined with Week 4's finding that age has a non-monotonic (U-shaped) relationship to decline, this suggests age is a genuine, if modest, multivariate signal rather than a leak in disguise.

In [ ]:
print("Features used:", feature_cols)
print("\nAttack checklist:")
print("1. Timeline: features from March 1-31, label ('declined') from within-March half-split — same window, flagged as a known limitation (not a strict past→future gap).")
print("2. Label-derived features in X?", any(c in feature_cols for c in ["pct_change", "declined"]))
print("3. Product flags in X?", any(c in feature_cols for c in ["health_score", "priority_score", "action_type"]))
print("4. Split grouped by client?", "Yes (Section 2)")
print("5. Base rate:", y.mean().round(3), "vs Random Forest precision@50:", round(precision_at_k(score_grouped, y_te_g.values, 50), 3))

# train-without test on the suspect signal (content_age_days), since Week 5 flagged it as unexpectedly top-importance
X_without_age = X_tr_g.drop(columns=["content_age_days"])
X_test_without_age = X_te_g.drop(columns=["content_age_days"])
rf_no_age = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42)
rf_no_age.fit(X_without_age, y_tr_g)
score_no_age = rf_no_age.predict_proba(X_test_without_age)[:, 1]
print("\nPrecision@50 WITH content_age_days:", round(precision_at_k(score_grouped, y_te_g.values, 50), 3))
print("Precision@50 WITHOUT content_age_days:", round(precision_at_k(score_no_age, y_te_g.values, 50), 3))

Features used: ['impressions_month', 'avg_position', 'ctr', 'content_age_days', 'word_count', 'search_volume', 'competition']

Attack checklist:
1. Timeline: features from March 1-31, label ('declined') from within-March half-split — same window, flagged as a known limitation (not a strict past→future gap).
2. Label-derived features in X? False
3. Product flags in X? False
4. Split grouped by client? Yes (Section 2)
5. Base rate: 0.377 vs Random Forest precision@50: 0.66

Precision@50 WITH content_age_days: 0.66
Precision@50 WITHOUT content_age_days: 0.68


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original (too bold): "Random Forest beats the hand-written rule by 6x."
Rewritten: "On this held-out, client-grouped sample, Random Forest's precision@50 was directionally and substantially higher than the baseline rule's (0.68 vs 0.10) — an observed result on a within-month proxy label, not a validated production outcome."

Original (too bold): "The model found the real pattern the rule missed."
Rewritten: "The model's precision@50 (0.66, grouped-split) was well above the base rate (0.377) and the baseline rule (0.10) — a measured, directional result on this sample. It does not establish that the underlying pattern is causal or stable across future months."

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.